# 04 · Comparación

Calcula la **ganancia en la banda** (promedio entre 3.000 y 6.000 envíos) por mes y por semilla, y compara:
- en el **`main`**: contra `AC` (201906–10) y contra A-consigna (201908–10). Si el `main` es `PAQ_P3`, comprueba que da lo del `22` (+9,00 y +12,60 M por mes);
- en un **branch**: contra las predicciones guardadas del `main` (no lo vuelve a correr), y aplica la **regla de merge**: `d` > 0 y al menos 3 de 5 semillas a favor, en 201906–10. Si el branch declara `'eligio_entre_varias': True`, se juzga solo en 201909–10.

Solo lee predicciones hechas con la configuración actual (`config.json` de cada corrida).

El intervalo suma dos errores: el de clientes (bootstrap pareado) y el de semilla. **Todo es exploratorio**: los meses de 2019 ya se usaron para decidir.

In [ ]:
import importlib
import time

import numpy as np
import pandas as pd

import warnings

import parametros as P
import funciones as F

warnings.filterwarnings('ignore', message='.*ChainedAssignmentError.*')   # falso positivo de pandas 2.2 con Python 3.14

importlib.reload(P)
importlib.reload(F)
P.RESULTADOS.mkdir(parents=True, exist_ok=True)
print('Corrida:', P.CORRIDA['nombre'], '·', len(P.CORRIDA['bloques']), 'bloques')

In [ ]:
from sklearn.metrics import roc_auc_score

info = F.leer_filas(P)
SEMILLAS = F.semillas(P, P.N_SEMILLAS)
E = F.Evaluacion(P, info, SEMILLAS)
configs = {c['nombre']: c for c in F.configuraciones(P)}
firmas = {k: F.revisar_config(P, c, crear=False) for k, c in configs.items()}   # solo predicciones de esta configuración
COMPARACIONES = F.comparaciones(P)
corridas = list(dict.fromkeys(x for c, contra, _ in COMPARACIONES for x in (c, contra)))
for c in corridas:
    E.leer(c, configs[c]['meses'])
print('Predicciones leídas:', {c: firmas[c] for c in corridas})

## 1. Ganancia por mes y por semilla

In [ ]:
filas = [{'corrida': c, 'foto_mes': m, 'semilla': s, 'ganancia_banda_M': round(float(E.banda_mes(c, m)[i]), 4),
          'auc': round(roc_auc_score(E.es_b2[E.es_mes(m)], E.puntaje[(c, m)][i]), 6), 'firma_config': firmas[c]}
         for c in corridas for m in configs[c]['meses'] for i, s in enumerate(SEMILLAS)]
ganancias = pd.DataFrame(filas)
for c in corridas:                     # en el main se guardan main, AC y A_CONSIGNA; en un branch, solo el branch
    if c == P.CORRIDA['nombre'] or P.CORRIDA['nombre'] == 'main':
        ganancias[ganancias['corrida'] == c].to_csv(P.RESULTADOS / f'{c}_ganancias.csv', index=False)
ganancias.pivot_table(index='corrida', columns='foto_mes', values='ganancia_banda_M').round(2)

## 2. Comparaciones

In [ ]:
comparacion = pd.DataFrame([E.comparar(c, contra, meses, P.SEMILLA_BOOTSTRAP + j)
                            for j, (c, contra, meses) in enumerate(COMPARACIONES)])
if P.CORRIDA['nombre'] != 'main':
    comparacion['eligio_entre_varias'] = bool(P.CORRIDA.get('eligio_entre_varias'))
    comparacion['veredicto'] = ['PASA' if F.regla_merge(r, P) else 'NO PASA' for r in comparacion.to_dict('records')]
comparacion['firma_config'] = [firmas[c] for c in comparacion['config']]
comparacion['firma_contra'] = [firmas[c] for c in comparacion['contra']]
comparacion.round(4).to_csv(P.RESULTADOS / f'{P.CORRIDA["nombre"]}_comparacion.csv', index=False)
for r in comparacion.to_dict('records'):
    print(f'{r["config"]} − {r["contra"]} ({r["meses"]}): {r["d_M"]:+.2f} M por mes '
          f'({r["ic95_inf_M"]:.2f} a {r["ic95_sup_M"]:.2f}) · ensamble {r["d_ensamble_M"]:+.2f} · '
          f'semillas a favor {r["semillas_a_favor"]} de {len(SEMILLAS)}')
    print('   por mes:', {k[2:]: round(v, 2) for k, v in r.items() if k.startswith('d_2') and not np.isnan(v)})

## 3. Veredicto

In [ ]:
if P.CORRIDA['nombre'] == 'main':
    if F.estrategia(P.MAIN) == F.estrategia(P.PAQ_P3):
        for r in comparacion.to_dict('records'):
            esperado = P.ESPERADO_PAQ_P3[r['contra']]
            for campo in ('d_M', 'ic95_inf_M', 'ic95_sup_M'):
                assert abs(r[campo] - esperado[campo]) < 0.005, (r['contra'], campo, r[campo])
            for m, v in esperado['por_mes'].items():
                assert abs(r[f'd_{m}'] - v) < 0.005, (r['contra'], m, r[f'd_{m}'])
            assert r['semillas_a_favor'] == esperado['semillas_a_favor']
        print('MAIN VERIFICADO: es PAQ_P3 y reproduce el 22 (+9,00 M contra AC y +12,60 M contra A-consigna).')
    else:
        print('MAIN NUEVO (después de un merge): no es PAQ_P3, así que no se compara con el 22. '
              'Sus huellas, ganancias y comparaciones quedan como la referencia nueva: versionarlas.')
else:
    r = comparacion.iloc[0].to_dict()
    eligio = ', eligió entre varias configuraciones' if r['eligio_entre_varias'] else ''
    print(f'REGLA DE MERGE ({r["meses"]}{eligio}): d = {r["d_M"]:+.2f} M, {r["semillas_a_favor"]} de {len(SEMILLAS)} '
          f'semillas a favor → {"PASA: se puede hacer merge" if r["veredicto"] == "PASA" else "NO PASA: queda el main"}')
print('EXPLORATORIO: no cambia el resultado de la regla pre-registrada.')

## 4. Figuras

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter

TINTA, GRILLA, AZUL = '#52514e', '#e1e0d9', '#2a78d6'
plt.rcParams.update({'figure.facecolor': '#fcfcfb', 'axes.facecolor': '#fcfcfb', 'axes.edgecolor': '#c3c2b7',
                     'axes.labelcolor': TINTA, 'xtick.color': TINTA, 'ytick.color': TINTA, 'axes.grid': True,
                     'grid.color': GRILLA, 'grid.linewidth': 0.6, 'axes.spines.top': False, 'axes.spines.right': False,
                     'font.size': 13, 'axes.titlesize': 15})
coma = FuncFormatter(lambda v, _: f'{v:,.1f}'.replace(',', '_').replace('.', ',').replace('_', '.'))
miles = FuncFormatter(lambda v, _: f'{v:,.0f}'.replace(',', '.'))
carpeta = P.RESULTADOS / 'figuras'
carpeta.mkdir(parents=True, exist_ok=True)
HASTA = 12_000                                   # envíos que muestra la figura 2

r = comparacion.iloc[0]
meses = [int(k[2:]) for k in r.index if k.startswith('d_2') and not np.isnan(r[k])]
fig, ax = plt.subplots(figsize=(12.8, 7.2))
ax.axhline(0, color=TINTA, lw=1.2)
x = [str(m) for m in meses] + ['media']
ax.bar(x[:-1], [r[f'd_{m}'] for m in meses], color=AZUL, width=0.5)
ax.errorbar([x[-1]], [r.d_M], yerr=[[r.d_M - r.ic95_inf_M], [r.ic95_sup_M - r.d_M]], fmt='o', color=AZUL, ms=9, capsize=6, lw=2)
ax.yaxis.set_major_formatter(coma)
ax.set_ylabel('Diferencia en la banda (M$ por mes)')
ax.grid(axis='x', visible=False)
fig.suptitle(f'{r.config} − {r.contra}, por mes ({len(SEMILLAS)} semillas)', x=0.02, ha='left', fontsize=16)
ax.set_title('La media lleva el IC 95 %', loc='left', color=TINTA, fontsize=13)
fig.tight_layout(rect=(0, 0, 1, 0.95))
fig.savefig(carpeta / f'{P.CORRIDA["nombre"]}_1_por_mes.png', dpi=150)
plt.show()

fig, ax = plt.subplots(figsize=(12.8, 7.2))
visibles = []
for c, color in ((r.config, AZUL), (r.contra, TINTA)):
    cv = E.curva_media(c, meses)
    visibles.append(cv[:HASTA])
    ax.plot(np.arange(1, P.LARGO_CURVA + 1), cv, color=color, lw=2, label=c)
ax.axvspan(*P.BANDA, color=GRILLA, alpha=0.5, label='banda')
ax.set_xlim(0, HASTA)
alto, bajo = max(v.max() for v in visibles), min(0, min(v.min() for v in visibles))
ax.set_ylim(bajo, alto * 1.08)
ax.xaxis.set_major_formatter(miles)
ax.yaxis.set_major_formatter(coma)
ax.set_xlabel('Envíos')
ax.set_ylabel('Ganancia (M$ por mes, media de meses y semillas)')
ax.legend(loc='lower right')
fig.suptitle('Ganancia según la cantidad de envíos', x=0.02, ha='left', fontsize=16)
fig.tight_layout(rect=(0, 0, 1, 0.95))
fig.savefig(carpeta / f'{P.CORRIDA["nombre"]}_2_curva.png', dpi=150)
plt.show()